In [ ]:
%matplotlib inline

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine, text
import os
from dotenv import load_dotenv

plt.style.use('ggplot')
sns.set_palette("husl")

print("Библиотеки загружены!")

In [ ]:
import sys
print(sys.executable)
print(sys.version)

In [ ]:
import sys
!{sys.executable} -m pip show psycopg2-binary

In [ ]:
import psycopg2
print("Версия psycopg2:", psycopg2.__version__)

In [ ]:
engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

In [ ]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT version();"))
    row = result.fetchone()
    print("✅ Подключение успешно!")
    print(row[0])

In [ ]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
print("✅ Импорты OK")

In [ ]:
load_dotenv()

DB_USER = os.getenv("DB_USER")
DB_PASS = os.getenv("DB_PASS")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

with engine.connect() as conn:
    result = conn.execute(text("SELECT version();"))
    row = result.fetchone()
    print("✅ Подключение успешно!")
    print(row[0])

In [ ]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv()

engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('DB_USER')}:{os.getenv('DB_PASS')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)

with engine.connect() as conn:
    print("✅ Подключение успешно!")
    print(conn.execute(text("SELECT version();")).fetchone()[0])

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

print("Библиотеки для анализа загружены")

In [ ]:
df = pd.read_csv("../data/HeartDiseaseTrain-Test.csv")
print("Размер:", df.shape)
df.head()

In [ ]:
df.dtypes

In [ ]:
for col in df.select_dtypes(include='object').columns:
    print(f"{col}: {df[col].unique()}")

In [ ]:
# Кодируем текстовые столбцы в числа

df['sex'] = df['sex'].map({'Female': 0, 'Male': 1})

df['chest_pain_type'] = df['chest_pain_type'].map({
    'Typical angina': 0,
    'Atypical angina': 1,
    'Non-anginal pain': 2,
    'Asymptomatic': 3
})

df['fasting_blood_sugar'] = df['fasting_blood_sugar'].map({
    'Lower than 120 mg/ml': 0,
    'Greater than 120 mg/ml': 1
})

df['rest_ecg'] = df['rest_ecg'].map({
    'Normal': 0,
    'ST-T wave abnormality': 1,
    'Left ventricular hypertrophy': 2
})

df['exercise_induced_angina'] = df['exercise_induced_angina'].map({
    'No': 0, 'Yes': 1
})

df['slope'] = df['slope'].map({
    'Upsloping': 0, 'Flat': 1, 'Downsloping': 2
})

df['vessels_colored_by_flourosopy'] = df['vessels_colored_by_flourosopy'].map({
    'Zero': 0, 'One': 1, 'Two': 2, 'Three': 3, 'Four': 4
})

df['thalassemia'] = df['thalassemia'].map({
    'Normal': 0, 'Fixed Defect': 1, 'Reversable Defect': 2, 'No': 3
})

print("✅ Кодировка завершена")
df.dtypes

In [ ]:
df = df.rename(columns={
    'chest_pain_type': 'cp',
    'resting_blood_pressure': 'trestbps',
    'cholestoral': 'chol',
    'fasting_blood_sugar': 'fbs',
    'rest_ecg': 'restecg',
    'Max_heart_rate': 'thalach',
    'exercise_induced_angina': 'exang',
    'vessels_colored_by_flourosopy': 'ca',
    'thalassemia': 'thal'
})

print("✅ Столбцы переименованы")
print(df.columns.tolist())

In [ ]:
df.isnull().sum()

In [ ]:
df.head()


In [ ]:
df = df.drop_duplicates()
print("После удаления дубликатов:", df.shape)

In [ ]:
df.to_sql("patients", engine, if_exists="replace", index=False)
print("✅ Таблица patients загружена в Postgres")

In [ ]:
pd.read_sql("SELECT * FROM patients LIMIT 5", engine)

In [ ]:
pd.read_sql("SELECT COUNT(*) AS n, MIN(age) AS min_age, MAX(age) AS max_age FROM patients", engine)

In [ ]:
sql_script = """
CREATE TABLE IF NOT EXISTS sex_dict (
    code INT PRIMARY KEY,
    label TEXT NOT NULL
);

INSERT INTO sex_dict (code, label) VALUES
    (0, 'Female'), (1, 'Male')
ON CONFLICT (code) DO NOTHING;

CREATE TABLE IF NOT EXISTS cp_dict (
    code INT PRIMARY KEY,
    label TEXT NOT NULL
);

INSERT INTO cp_dict (code, label) VALUES
    (0, 'Typical angina'), (1, 'Atypical angina'),
    (2, 'Non-anginal pain'), (3, 'Asymptomatic')
ON CONFLICT (code) DO NOTHING;
"""

from sqlalchemy import text

with engine.begin() as conn:
    conn.execute(text(sql_script))

print("✅ Справочники созданы")

In [ ]:
print("sex_dict:")
print(pd.read_sql("SELECT * FROM sex_dict", engine))

print("\ncp_dict:")
print(pd.read_sql("SELECT * FROM cp_dict", engine))

In [ ]:
query1 = """
SELECT 
    COUNT(*) AS total_patients,
    ROUND(AVG(age)::numeric, 1) AS avg_age,
    ROUND(AVG(chol)::numeric, 1) AS avg_chol,
    ROUND(AVG(trestbps)::numeric, 1) AS avg_bp,
    ROUND(AVG(thalach)::numeric, 1) AS avg_max_hr
FROM patients;
"""
pd.read_sql(query1, engine)

In [ ]:
query2 = """
SELECT 
    target,
    COUNT(*) AS count,
    ROUND(AVG(age)::numeric, 1) AS avg_age,
    ROUND(AVG(chol)::numeric, 1) AS avg_chol,
    ROUND(AVG(thalach)::numeric, 1) AS avg_thalach
FROM patients
GROUP BY target
ORDER BY target;
"""
pd.read_sql(query2, engine)

In [ ]:
query3 = """
SELECT 
    s.label AS sex,
    COUNT(*) AS total,
    SUM(p.target) AS sick,
    ROUND(AVG(p.target) * 100, 1) AS disease_percent
FROM patients p
JOIN sex_dict s ON p.sex = s.code
GROUP BY s.label
ORDER BY s.label;
"""
pd.read_sql(query3, engine)

In [ ]:
query4 = """
WITH age_groups AS (
    SELECT 
        *,
        CASE 
            WHEN age < 40 THEN '1. Under 40'
            WHEN age < 55 THEN '2. 40-54'
            WHEN age < 70 THEN '3. 55-69'
            ELSE '4. 70+'
        END AS age_group
    FROM patients
)
SELECT 
    age_group,
    COUNT(*) AS total,
    ROUND(AVG(target) * 100, 1) AS disease_percent
FROM age_groups
GROUP BY age_group
ORDER BY age_group;
"""
pd.read_sql(query4, engine)

In [ ]:
print("В pandas:", df.shape)
print("В Postgres:", pd.read_sql("SELECT COUNT(*) AS n FROM patients", engine).iloc[0,0])
print("Пропуски в df:")
print(df.isnull().sum().sum())   # общее число NaN

In [ ]:
query5 = """
SELECT 
    age, sex, chol, trestbps, target,
    RANK() OVER (ORDER BY chol DESC) AS chol_rank
FROM patients
ORDER BY chol DESC
LIMIT 10;
"""
pd.read_sql(query5, engine)

In [ ]:
query6 = """
SELECT 
    c.label AS chest_pain,
    COUNT(*) AS total,
    ROUND(AVG(p.target) * 100, 1) AS disease_percent
FROM patients p
JOIN cp_dict c ON p.cp = c.code
GROUP BY c.label
ORDER BY disease_percent DESC;
"""
pd.read_sql(query6, engine)

In [ ]:
# 1. Распределение target
print("=" * 50)
print("Распределение target:")
print(df['target'].value_counts())
print()

# 2. Возраст по группам
print("=" * 50)
print("Возраст по группам:")
print(df.groupby('target')['age'].describe().round(1))
print()

# 3. Корреляции с target
print("=" * 50)
print("Корреляции с target (по убыванию):")
print(df.corr(numeric_only=True)['target'].sort_values(ascending=False).round(3))

In [ ]:
# Проверяем: у кого exang=1 (стенокардия при нагрузке) — какой target?
print("exang vs target:")
print(pd.crosstab(df['exang'], df['target'], margins=True))
print()

# У кого ca > 0 (поражённые сосуды) — какой target?
print("ca vs target:")
print(pd.crosstab(df['ca'], df['target'], margins=True))
print()

# У кого oldpeak > 0 — какой target?
print("oldpeak > 0 vs target:")
print(pd.crosstab(df['oldpeak'] > 0, df['target'], margins=True))

In [ ]:
# Создаём «человеческую» целевую переменную
df['disease'] = 1 - df['target']   # 1 = болен, 0 = здоров

# Проверяем: теперь больные должны быть старше, иметь больше поражённых сосудов и т.д.
print("Проверка после перекодировки:")
print(df.groupby('disease')[['age', 'chol', 'thalach', 'oldpeak', 'ca', 'exang']].mean().round(2))
print()
print("Распределение disease:")
print(df['disease'].value_counts())

In [ ]:
df.to_sql("patients", engine, if_exists="replace", index=False)
print("✅ Перезалито с колонкой disease")

In [ ]:
pd.read_sql("SELECT disease, COUNT(*) FROM patients GROUP BY disease", engine)

In [ ]:
query1 = """
SELECT 
    COUNT(*) AS total_patients,
    ROUND(AVG(age)::numeric, 1) AS avg_age,
    ROUND(AVG(chol)::numeric, 1) AS avg_chol,
    ROUND(AVG(trestbps)::numeric, 1) AS avg_bp,
    ROUND(AVG(thalach)::numeric, 1) AS avg_max_hr
FROM patients;
"""
pd.read_sql(query1, engine)

In [ ]:
query2 = """
SELECT 
    disease,
    COUNT(*) AS count,
    ROUND(AVG(age)::numeric, 1) AS avg_age,
    ROUND(AVG(chol)::numeric, 1) AS avg_chol,
    ROUND(AVG(thalach)::numeric, 1) AS avg_thalach,
    ROUND(AVG(oldpeak)::numeric, 2) AS avg_oldpeak
FROM patients
GROUP BY disease
ORDER BY disease;
"""
pd.read_sql(query2, engine)

In [ ]:
query3 = """
SELECT 
    s.label AS sex,
    COUNT(*) AS total,
    SUM(p.disease) AS sick,
    ROUND(AVG(p.disease) * 100, 1) AS disease_percent
FROM patients p
JOIN sex_dict s ON p.sex = s.code
GROUP BY s.label
ORDER BY s.label;
"""
pd.read_sql(query3, engine)

In [ ]:
query4 = """
WITH age_groups AS (
    SELECT 
        *,
        CASE 
            WHEN age < 40 THEN '1. Under 40'
            WHEN age < 55 THEN '2. 40-54'
            WHEN age < 70 THEN '3. 55-69'
            ELSE '4. 70+'
        END AS age_group
    FROM patients
)
SELECT 
    age_group,
    COUNT(*) AS total,
    ROUND(AVG(disease) * 100, 1) AS disease_percent
FROM age_groups
GROUP BY age_group
ORDER BY age_group;
"""
pd.read_sql(query4, engine)

In [ ]:
query5 = """
SELECT 
    age, sex, chol, trestbps, disease,
    RANK() OVER (ORDER BY chol DESC) AS chol_rank
FROM patients
ORDER BY chol DESC
LIMIT 10;
"""
pd.read_sql(query5, engine)

In [ ]:
query6 = """
SELECT 
    c.label AS chest_pain,
    COUNT(*) AS total,
    ROUND(AVG(p.disease) * 100, 1) AS disease_percent
FROM patients p
JOIN cp_dict c ON p.cp = c.code
GROUP BY c.label
ORDER BY disease_percent DESC;
"""
pd.read_sql(query6, engine)

In [ ]:
# 1. groupby — среднее по группам
print("Средний возраст по disease:")
print(df.groupby('disease')['age'].mean().round(2))
print()

# 2. corr — корреляции
print("Корреляции с disease:")
print(df.corr(numeric_only=True)['disease'].sort_values(ascending=False).round(3))
print()

# 3. crosstab — сопряжённость
print("exang vs disease:")
print(pd.crosstab(df['exang'], df['disease'], margins=True))

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(df[df['disease']==0]['age'], bins=15, alpha=0.6, label='Healthy', color='green')
plt.hist(df[df['disease']==1]['age'], bins=15, alpha=0.6, label='Sick', color='red')
plt.xlabel('Возраст')
plt.ylabel('Количество пациентов')
plt.title('Распределение возраста: больные vs здоровые')
plt.legend()
plt.savefig('../figures/01_age_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.boxplot(x='disease', y='chol', data=df)
plt.xticks([0, 1], ['Healthy', 'Sick'])
plt.xlabel('Группа')
plt.ylabel('Холестерин (мг/дл)')
plt.title('Холестерин у больных и здоровых')
plt.savefig('../figures/02_chol_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(12, 10))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Корреляционная матрица признаков')
plt.savefig('../figures/03_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
sex_data = df.groupby('sex')['disease'].mean() * 100

plt.figure(figsize=(8, 6))
plt.bar(['Female', 'Male'], sex_data.values, color=['pink', 'steelblue'])
plt.ylabel('Процент больных')
plt.title('Заболеваемость по полу')
for i, v in enumerate(sex_data.values):
    plt.text(i, v + 1, f'{v:.1f}%', ha='center')
plt.savefig('../figures/04_sex_rate.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(df[df['disease']==0]['age'], df[df['disease']==0]['thalach'], 
            alpha=0.6, label='Healthy', color='green')
plt.scatter(df[df['disease']==1]['age'], df[df['disease']==1]['thalach'], 
            alpha=0.6, label='Sick', color='red')
plt.xlabel('Возраст')
plt.ylabel('Макс. пульс')
plt.title('Возраст vs Макс. пульс')
plt.legend()
plt.savefig('../figures/05_age_vs_thalach.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
df['age_group'] = pd.cut(df['age'], 
                          bins=[0, 40, 55, 70, 100], 
                          labels=['<40', '40-54', '55-69', '70+'],
                          right=False)

age_stats = df.groupby('age_group', observed=True)['disease'].mean() * 100

plt.figure(figsize=(10, 6))
plt.bar(age_stats.index.astype(str), age_stats.values, color='coral')
plt.ylabel('Процент больных')
plt.title('Заболеваемость по возрастным группам')
for i, v in enumerate(age_stats.values):
    plt.text(i, v + 1, f'{v:.1f}%', ha='center')
plt.savefig('../figures/06_age_group_rate.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
cp_labels = {0: 'Typical angina', 1: 'Atypical angina', 
             2: 'Non-anginal pain', 3: 'Asymptomatic'}
cp_data = df.groupby('cp')['disease'].mean() * 100

plt.figure(figsize=(10, 6))
bars = plt.bar([cp_labels[i] for i in cp_data.index], cp_data.values, color='teal')
plt.ylabel('Процент больных')
plt.title('Заболеваемость по типу боли в груди')
plt.xticks(rotation=20)
for i, v in enumerate(cp_data.values):
    plt.text(i, v + 1, f'{v:.1f}%', ha='center')
plt.savefig('../figures/07_chest_pain_rate.png', dpi=150, bbox_inches='tight')
plt.show()

# Выводы

## Основные инсайты

1. **Пол:** мужчины болеют в 2.2 раза чаще женщин (55.3% vs 25.0%).
2. **Возраст:** критическая группа — 55–69 лет (заболеваемость 60.4%).
3. **Пульс:** у больных максимальный пульс ниже (139.1 vs 158.4 уд/мин).
4. **Холестерин:** у больных выше (251.1 vs 242.6 мг/дл).
5. **ЭКГ (oldpeak):** у больных значение в 2.7 раза выше (1.59 vs 0.59).
6. **Тип боли:** типичная стенокардия — самый сильный предиктор (72.7%).
7. **Ключевые предикторы:** exang, oldpeak, ca, thalach, cp.

## Обнаруженная проблема данных

В исходном датасете метки `target` были **инвертированы**: 
`target=0` означал больного, `target=1` — здорового. 
Мы это обнаружили при анализе корреляций и исправили, 
создав корректную переменную `disease`.

## Ограничения

- Датасет небольшой (302 записи).
- Данные 1988 года (Кливленд) — не отражают современность.
- Нет данных об образе жизни (курение, питание, спорт).
- Корреляция ≠ причинность.

## Перспективы

- Проверка гипотез на современных данных.
- Построение ML-модели для предсказания риска.
- Расширение датасета факторами образа жизни.